### **Data Loading and Cleaning: ACLED**

This primary notebook shall take data from the Armed Conflict Location & Event Data (ACLED) database on political violence. It shall seek to analyse the distribution of state and non-state actors' violent acts, in turn drawing conclusions on the relationship between state and non-state violence.

This notebook will clean and wrangle the large initial database into a dataframe sufficient for analysis in notebook **02_exploration.ipynb**.

Import pandas library & OS for file management.

In [2]:
# Import both, apply alias for pandas.
import pandas as pd
import os

Gdown loads large Google files (datset is 130Mb).

In [3]:
# Download gdown.
!pip install gdown

ACLED Dataset = disaggregated conflict data project, codes political violence and
protest events. Data collected through systematic review of local/national/international media reports, NGO publications, govt sources.
Each event coded and assigned attributes of:

- Date (to the day; *time_precision* records date confidence)
- Event type (*battles*, *explosions/remote violence*, *protests*, *riots*,
  *violence against civilians*, *strategic developments*)
- Actors (*primary* and *secondary*, with interaction code)
- Location (*country*, *admin1–3*, *coordinates*, *geo_precision*)
- Fatalities (recorded where reported; acknowledged to be lower than truth)

ACLED distinguishes between state-based actors (*governments*, *military*, *police*)
and non-state actors (*rebel groups*, *militias*, *criminal organisations*). Distinguishing between the two forms basis of comparative analysis in this project.

Sourced from ACLED Codebook 2025 (dataset goes up to 21/04/2025 [date imported]).

Download ACLED dataset from ACLED data export tool.

Import data from 01/01/2018 onwards, with ACLED backdating regional data collection to begin from this date (no prior data exists). 7 years shall provide sufficient overview of trends beyond small regional/national shocks.

Only import data with *South America* value in *region*, with regional specification allowing more in-depth analysis.

Do not import data tagged under *Strategic Developments*, as these are categorised as non-violent in Codebook, and not relevant to analysis.

In [4]:
# Import gdown.
import gdown

# Load downloaded dataset from Drive location.
url = 'https://drive.google.com/uc?id=1624LkOTd02QV2sTgMvXb08_3PJBHVeMV'
gdown.download(url, 'acled_data.csv', quiet=False)


Downloading...
From (original): https://drive.google.com/uc?id=1624LkOTd02QV2sTgMvXb08_3PJBHVeMV
From (redirected): https://drive.google.com/uc?id=1624LkOTd02QV2sTgMvXb08_3PJBHVeMV&confirm=t&uuid=c2eb1b9f-b3c8-4c12-95cb-c4ccdb78631d
To: /content/acled_data.csv
100%|██████████| 136M/136M [00:02<00:00, 60.9MB/s]


'acled_data.csv'

Display full dataset.

In [5]:
# Read dataset and display with headings.
df = pd.read_csv('acled_data.csv')
df.head()

,event_id_cnty,event_date,year,time_precision,disorder_type,event_type,sub_event_type,actor1,assoc_actor_1,inter1,...,location,latitude,longitude,geo_precision,source,source_scale,notes,fatalities,tags,timestamp
0,ARG1022,2019-11-13,2019,1,Demonstrations,Protests,Peaceful protest,Protesters (Argentina),ATE: Association of State Workers; CTA: Argent...,Protesters,...,Rosario,-32.9528,-60.6473,1,Sin Mordaza; La Capital (Argentina),Subnational,"On 13 November 2019, in Rosario (Santa Fe), he...",0,crowd size=no report,1582839774
1,ARG1038,2019-11-16,2019,1,Demonstrations,Protests,Peaceful protest,Protesters (Argentina),LGBTQ+ (Argentina),Protesters,...,Santiago del Estero,-27.7984,-64.2728,1,El Liberal; Diario Panorama,Subnational-International,"On 16 November 2019, in Santiago del Estero (S...",0,crowd size=no report,1618524813
2,ARG1276,2020-01-06,2020,1,Demonstrations,Protests,Peaceful protest,Protesters (Argentina),NaN,Protesters,...,Caleta Olivia,-46.4445,-67.5223,1,Diario Cronica (Argentina); Tiempo Sur (Argent...,Subnational-National,"On 6 January 2020, in Caleta Olivia (Santa Cru...",0,crowd size=no report,1582839774
3,ARG1277,2020-01-06,2020,1,Demonstrations,Protests,Peaceful protest,Protesters (Argentina),Labor Group (Argentina),Protesters,...,Caleta Olivia,-46.4445,-67.5223,1,Tiempo Sur (Argentina); Diario Cronica (Argent...,Subnational-National,"On 6 January 2020, in Caleta Olivia (Santa Cru...",0,crowd size=no report,1582839774
4,ARG129,2019-02-01,2019,1,Demonstrations,Protests,Peaceful protest,Protesters (Argentina),Labor Group (Argentina),Protesters,...,Ezeiza,-34.8531,-58.5222,1,La Union (Argentina); Indymedia Argentina,Subnational-National,"On 1 February 2019, in Ezeiza, Buenos Aires, e...",0,crowd size=-no report,1618524821


Columns exceed maximum display, must remove columns that are redundant for quantitative analysis (per Codebook definitions) or with a high percentage of null responses to ensure full display. Display in table to check.

In [6]:
# Assign which columns to keep.
cols_to_keep = ['event_date', 'year', 'time_precision', 'disorder_type','event_type', 'sub_event_type', 'actor1', 'inter1','actor2', 'inter2', 'interaction', 'country', 'admin1','admin2', 'location', 'latitude', 'longitude','geo_precision', 'source_scale', 'fatalities']

# Display new table.
df = pd.read_csv('acled_data.csv', usecols=cols_to_keep)
df.head()

,event_date,year,time_precision,disorder_type,event_type,sub_event_type,actor1,inter1,actor2,inter2,interaction,country,admin1,admin2,location,latitude,longitude,geo_precision,source_scale,fatalities
0,2019-11-13,2019,1,Demonstrations,Protests,Peaceful protest,Protesters (Argentina),Protesters,NaN,NaN,Protesters only,Argentina,Santa Fe,Rosario,Rosario,-32.9528,-60.6473,1,Subnational,0
1,2019-11-16,2019,1,Demonstrations,Protests,Peaceful protest,Protesters (Argentina),Protesters,NaN,NaN,Protesters only,Argentina,Santiago del Estero,Capital,Santiago del Estero,-27.7984,-64.2728,1,Subnational-International,0
2,2020-01-06,2020,1,Demonstrations,Protests,Peaceful protest,Protesters (Argentina),Protesters,NaN,NaN,Protesters only,Argentina,Santa Cruz,Deseado,Caleta Olivia,-46.4445,-67.5223,1,Subnational-National,0
3,2020-01-06,2020,1,Demonstrations,Protests,Peaceful protest,Protesters (Argentina),Protesters,NaN,NaN,Protesters only,Argentina,Santa Cruz,Deseado,Caleta Olivia,-46.4445,-67.5223,1,Subnational-National,0
4,2019-02-01,2019,1,Demonstrations,Protests,Peaceful protest,Protesters (Argentina),Protesters,NaN,NaN,Protesters only,Argentina,Buenos Aires,Jose M. Ezeiza,Ezeiza,-34.8531,-58.5222,1,Subnational-National,0


Columns removed: those with zero/little variance (*region, civilian_targeting*), those redundant given better equivalents (*iso, timestamp*), those with too many unique values (*notes, source*), those with little analytical use (*event_id_cnty, source*), those with a high % of null values (*admin3, assoc_actor_1, assoc_actor_2, civilian_targeting, tags*). Therefore, most null values are removed.


Assign state and non-state labels for ease of analysis; display these in updated table to check.




In [7]:
# Assign state or non-state labels.
state_keywords = ['government', 'state', 'military', 'police']
non_state_keywords = ['rebel', 'militia', 'criminal organisation', 'gang', 'rioter', 'protesters', 'civilian']

# Categorize actors.
def categorize_actor(actor_name):
    if pd.isna(actor_name):
        return None
    actor_name_lower = str(actor_name).lower()

    if any(keyword in actor_name_lower for keyword in state_keywords):
        return 'State'
    elif any(keyword in actor_name_lower for keyword in non_state_keywords):
        return 'Non-State'
    else:
        return 'Other'

# Apply categorization to actor1 and actor2 columns.
df['actor1_type'] = df['actor1'].apply(categorize_actor)
df['actor2_type'] = df['actor2'].apply(categorize_actor)

# Create columns indicating if event involves state or non-state actor.
df['is_state_actor_event'] = ((df['actor1_type'] == 'State') | (df['actor2_type'] == 'State')).astype(int)
df['is_non_state_actor_event'] = ((df['actor1_type'] == 'Non-State') | (df['actor2_type'] == 'Non-State')).astype(int)

# Display relevant columns.
df[['actor1', 'actor1_type', 'actor2', 'actor2_type', 'is_state_actor_event', 'is_non_state_actor_event']].head()

,actor1,actor1_type,actor2,actor2_type,is_state_actor_event,is_non_state_actor_event
0,Protesters (Argentina),Non-State,NaN,None,0,1
1,Protesters (Argentina),Non-State,NaN,None,0,1
2,Protesters (Argentina),Non-State,NaN,None,0,1
3,Protesters (Argentina),Non-State,NaN,None,0,1
4,Protesters (Argentina),Non-State,NaN,None,0,1


For some events, both actors involved are categorised as Other or NaN (with *NaN* referring to no assigned number/value) if undetermined to fit within state or non-state categories.

Remove these as they are void in comparing state and non-state. Display to check results, include raw datapoints to compare available data after wrangling.

In [8]:
# Filter out events where actor1_type and actor2_type are 'Other'.
df_filtered = df[~((df['actor1_type'] == 'Other') & (df['actor2_type'] == 'Other'))]

# Filter out events where actor1_type or actor2_type is 'Other' and the other is NaN
df_filtered = df_filtered[~(((df_filtered['actor1_type'] == 'Other') & (df_filtered['actor2_type'].isnull())) | ((df_filtered['actor2_type'] == 'Other') & (df_filtered['actor1_type'].isnull())))]

# Display original and filtered DataFrame shape to see number of removal.
print(f"Original DataFrame shape: {df.shape}")
print(f"Filtered DataFrame shape: {df_filtered.shape}")

display(df_filtered.head())

Original DataFrame shape: (214305, 24)
Filtered DataFrame shape: (209317, 24)


,event_date,year,time_precision,disorder_type,event_type,sub_event_type,actor1,inter1,actor2,inter2,...,location,latitude,longitude,geo_precision,source_scale,fatalities,actor1_type,actor2_type,is_state_actor_event,is_non_state_actor_event
0,2019-11-13,2019,1,Demonstrations,Protests,Peaceful protest,Protesters (Argentina),Protesters,NaN,NaN,...,Rosario,-32.9528,-60.6473,1,Subnational,0,Non-State,None,0,1
1,2019-11-16,2019,1,Demonstrations,Protests,Peaceful protest,Protesters (Argentina),Protesters,NaN,NaN,...,Santiago del Estero,-27.7984,-64.2728,1,Subnational-International,0,Non-State,None,0,1
2,2020-01-06,2020,1,Demonstrations,Protests,Peaceful protest,Protesters (Argentina),Protesters,NaN,NaN,...,Caleta Olivia,-46.4445,-67.5223,1,Subnational-National,0,Non-State,None,0,1
3,2020-01-06,2020,1,Demonstrations,Protests,Peaceful protest,Protesters (Argentina),Protesters,NaN,NaN,...,Caleta Olivia,-46.4445,-67.5223,1,Subnational-National,0,Non-State,None,0,1
4,2019-02-01,2019,1,Demonstrations,Protests,Peaceful protest,Protesters (Argentina),Protesters,NaN,NaN,...,Ezeiza,-34.8531,-58.5222,1,Subnational-National,0,Non-State,None,0,1


209,317 datapoints remain, more than enough for sufficient statistical overview. However, given data only runs to April for 2025, later analysis on year-on-year comparisons would be skewed, so 2025 events must be removed.

In [9]:
# Remove events from the year 2025
df_filtered = df_filtered[df_filtered['year'] != 2025]

# Display the new shape of the DataFrame
print(f"DataFrame shape after removing 2025 events: {df_filtered.shape}")

DataFrame shape after removing 2025 events: (201181, 24)


201181 datapoints remain - small decrease shows statistical irrelevance of limited 2025 data.

Now run a preliminary check on amount of state and amount of non-state tags to ensure there's sufficient data for both (i.e. not an analytically small amount for one or the other).

In [10]:
# Define state/non-state violence events from filtered DataFrame.
state_violence_events = df_filtered[df_filtered['is_state_actor_event'] == 1]
non_state_violence_events = df_filtered[df_filtered['is_non_state_actor_event'] == 1]

# Print shape to find amount for each tag.
print(f"Shape of state violence events: {state_violence_events.shape}")
print(f"Shape of non-state violence events: {non_state_violence_events.shape}")

Shape of state violence events: (58425, 24)
Shape of non-state violence events: (182220, 24)


There is sufficient data for both, with there still being 58,425 datapoints for the lower value, and already a clear trend in higher non-state violence than state violence.

Need to parse date inputs into datetime objects for later analysis, as string dates cannot calculate time difference, sort chronologically etc.

In [11]:
# Convert 'event_date' to datetime object.
df_filtered['event_date'] = pd.to_datetime(df_filtered['event_date'], format='%Y-%m-%d')

# Display quick iteration of table and check data type of column.
print(df_filtered[['event_date', 'year', 'time_precision', 'actor1_type', 'actor2_type', 'fatalities']].head())
print(f"Data type of 'event_date' in df_filtered: {df_filtered['event_date'].dtype}")

  event_date  year  time_precision actor1_type actor2_type  fatalities
0 2019-11-13  2019               1   Non-State        None           0
1 2019-11-16  2019               1   Non-State        None           0
2 2020-01-06  2020               1   Non-State        None           0
3 2020-01-06  2020               1   Non-State        None           0
4 2019-02-01  2019               1   Non-State        None           0
Data type of 'event_date' in df_filtered: datetime64[ns]


Additionally, within *fatalities* lies a considerable number of nulls ('no deaths reported') which must be assigned a value of 0 for analysis.

In [12]:
# Replace null values in 'fatalities' with 0
df_filtered['fatalities'] = df_filtered['fatalities'].fillna(0)

# Display the head of the DataFrame and check for nulls in 'fatalities' to confirm
print(df_filtered[['fatalities']].head())
print(f"Number of nulls in 'fatalities' column: {df_filtered['fatalities'].isnull().sum()}")

   fatalities
0           0
1           0
2           0
3           0
4           0
Number of nulls in 'fatalities' column: 0


To wrap up data wrangling, a few checks to ensure the proper parameters for subsequent analysis are met, and that all data is valid.

In [13]:
# Check year range is as expected.
assert df_filtered['year'].min() >= 2018 and df_filtered['year'].max() <= 2024, \
    "Year parameters are not within the 2005-2025 range."

# Check fatalities are non-negative.
assert (df_filtered['fatalities'] >= 0).all(), "Negative fatality values found."

# Check all rows have at least one valid actor type (for each event, either actor1_type or actor2_type (or both) must be 'State' or 'Non-State').
assert ((df_filtered['actor1_type'].isin(['State', 'Non-State'])) | \
        (df_filtered['actor2_type'].isin(['State', 'Non-State']))).all()

# If all assertions pass, print success message.
print("Parameter & validation checks passed.")

Parameter & validation checks passed.


Save filtered dataset to Google Drive folder.

In [14]:
from google.colab import drive
drive.mount('/content/drive')
df_filtered.to_pickle('/content/drive/MyDrive/df_filtered.pkl')


Mounted at /content/drive


Data wrangling is complete; can proceed with data analysis in notebook **02_exploration.ipnyb**.